# 05 · Export

Converts the trained checkpoint into the artefacts the Android app ships, then benchmarks and
verifies each one.

| variant | weights | activations | typical use |
|---|---|---|---|
| `float32` | float32 | float32 | reference — matches Keras exactly |
| `dynamic_range` | int8 | float32 | ~4× smaller, no calibration data needed |
| `int8` | int8 | int8 | smallest and fastest on phones; needs calibration clips |

Every variant is benchmarked (size, latency, arena estimate) and verified against the Keras model on
real clips, so a quantisation that damages accuracy is visible before it ships.

A checkpoint trained with mixed precision computes in float16, and TFLite has no float16 kernels for
`Conv2D` / `DepthwiseConv2dNative` / `Relu` — converting it directly fails with *"op is neither a
custom op nor a flex op"*. `export_all` handles this: it rebuilds the model in float32 first, which
is lossless because mixed precision keeps the master weights in float32 all along.

In [ ]:
#@title Setup — mount Drive, locate the project, install what is missing
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/MahmoudMabrok/SaloAleh.git"
REPO_BRANCH = os.environ.get("DHIKR_BRANCH", "main")
IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")


def sync_clone(target: Path) -> None:
    """Pull the latest code into an existing clone.

    Without this a runtime that cloned the repository earlier keeps running that
    old copy for the rest of the session, so fixes never arrive.
    """
    subprocess.run(
        ["git", "-C", str(target), "fetch", "--depth", "1", "origin", REPO_BRANCH], check=True
    )
    subprocess.run(
        ["git", "-C", str(target), "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"],
        check=True,
    )


def find_project_root() -> Path:
    """The folder that holds src/ and configs/ — cloned or updated as needed."""
    candidates = [Path(p) for p in [
        os.environ.get("DHIKR_PROJECT_ROOT", ""),
        "/content/DhikrSpeech",
        "/content/SaloAleh/DhikrSpeech",
        "/content/drive/MyDrive/DhikrSpeech",
    ] if p]
    candidates += [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "src" / "config.py").is_file() and (candidate / "configs" / "config.yaml").is_file():
            # Only ever update the throwaway clone this notebook created. A repo
            # you checked out yourself is left alone - resetting it would discard
            # whatever branch and local edits you are working on.
            if IN_COLAB and candidate.parent == Path("/content/SaloAleh"):
                try:
                    sync_clone(candidate.parent)
                except Exception as error:
                    print("could not update the clone, using it as is:", error)
            return candidate.resolve()
    if IN_COLAB:
        target = Path("/content/SaloAleh")
        print("cloning", REPO_URL, "@", REPO_BRANCH)
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(target)],
            check=True,
        )
        return (target / "DhikrSpeech").resolve()
    raise FileNotFoundError(
        "DhikrSpeech project not found. Set DHIKR_PROJECT_ROOT, or copy the "
        "DhikrSpeech folder to /content or to your Drive."
    )


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# A kernel that already imported src/ keeps the old modules even after the clone
# is updated, so drop them and let the imports below load the new code.
stale = [name for name in list(sys.modules) if name == "src" or name.startswith("src.")]
for name in stale:
    del sys.modules[name]

for module_name, package in [
    ("librosa", "librosa"),
    ("soundfile", "soundfile"),
    ("yaml", "PyYAML"),
    ("sklearn", "scikit-learn"),
    ("soxr", "soxr"),
]:
    if importlib.util.find_spec(module_name) is None:
        print("installing", package)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)

from src.config import load_config

CONFIG_PATH = PROJECT_ROOT / "configs" / "config.yaml"
config = load_config(CONFIG_PATH)
config.paths.ensure_dirs()

revision = subprocess.run(
    ["git", "-C", str(PROJECT_ROOT), "log", "-1", "--format=%h %s"],
    capture_output=True, text=True,
).stdout.strip()

print("project root :", PROJECT_ROOT)
print("code version :", revision or "(not a git checkout)")
print("config       :", CONFIG_PATH)
if stale:
    print("note         : reloaded %d cached src modules — re-run this notebook "
          "from the top so every stage uses the new code" % len(stale))
print()
print(config.summary())


## 1 · Load the model and rebuild the front-end

In [ ]:
import numpy as np
import pandas as pd

from src.dataset import (
    class_names_from_manifest, filter_split, load_manifest, load_phrases, make_tf_dataset,
)
from src.features import FeatureStats, LogMelExtractor
from src.trainer import load_trained_model

RUN_NAME = config.model.name

paths = config.paths
records = load_manifest(paths.manifest_path)
class_names = class_names_from_manifest(records)

checkpoint = paths.checkpoints_path / RUN_NAME / "best_model.keras"
model = load_trained_model(checkpoint)

stats_path = paths.processed_path / "feature_stats.json"
stats = FeatureStats.load(stats_path) if (
    config.features.normalize == "global" and stats_path.is_file()
) else None
extractor = LogMelExtractor(config.features, config.audio.sample_rate, stats=stats)

print("checkpoint :", checkpoint)
print("classes    :", len(class_names))
print("input shape:", config.input_shape)


## 2 · Calibration and verification clips

INT8 quantisation measures activation ranges from real features, so calibration comes from the
**training** split. Verification compares TFLite against Keras on **held-out** clips.

In [ ]:
from src.export import collect_features

train_records = filter_split(records, "train")
holdout_records = filter_split(records, "test") or filter_split(records, "val")

calibration_dataset = make_tf_dataset(
    train_records, config, extractor, training=False, batch_size=32, shuffle=False
)
calibration_features = collect_features(
    calibration_dataset, config.export.representative_samples
)

verification_features = None
if holdout_records:
    verification_dataset = make_tf_dataset(
        holdout_records, config, extractor, training=False, batch_size=32, shuffle=False
    )
    verification_features = collect_features(verification_dataset, 200)

print("calibration clips :", calibration_features.shape)
print("verification clips:", None if verification_features is None else verification_features.shape)


## 3 · Export, benchmark, verify

Conversion failures are isolated per variant: if INT8 fails, the other two are still produced.

In [ ]:
from src.export import export_all

metrics_payload = {}
evaluation_json = paths.reports_path / "evaluation.json"
if evaluation_json.is_file():
    import json

    payload = json.loads(evaluation_json.read_text(encoding="utf-8"))
    metrics_payload = {
        "accuracy": payload.get("accuracy"),
        "macro": payload.get("macro"),
        "num_samples": payload.get("num_samples"),
    }

phrases = {phrase.id: phrase.text for phrase in load_phrases(paths.phrases_path)}

bundle = export_all(
    model=model,
    config=config,
    class_names=class_names,
    frontend=extractor.metadata(),
    calibration_features=calibration_features,
    verification_features=verification_features,
    phrases=phrases,
    metrics=metrics_payload,
)

print()
print(bundle.table())


## 4 · Compare the variants

`expected_android_ms` is the measured latency multiplied by `export.android_latency_factor` — an
estimate for comparing variants, not a measurement. Measure on a real device before quoting it.

In [ ]:
from dataclasses import asdict

from src import visualization as viz

benchmarks = [item.benchmark for item in bundle.models if item.benchmark]
frame = pd.DataFrame([asdict(item) for item in benchmarks])
display(frame[[
    "name", "size_kb", "mean_latency_ms", "median_latency_ms", "p95_latency_ms",
    "arena_estimate_kb", "expected_android_ms", "input_dtype", "output_dtype",
]].round(3))

if benchmarks:
    figure = viz.plot_benchmark(benchmarks)
    viz.save_figure(figure, paths.reports_path / "05_benchmark.png")

verifications = [item.verification for item in bundle.models if item.verification]
if verifications:
    display(pd.DataFrame([asdict(item) for item in verifications]).round(5))
    for item in verifications:
        if not item.passed:
            print("WARNING: %s disagrees with the Keras model — do not ship it "
                  "without checking accuracy in 04_evaluation" % item.name)


## 5 · Front-end parameters for Android

The model takes log mel features, not raw audio, so the Android side must produce **identical**
features. These files pin that contract:

* `model_meta.json` — sample rate, clip length, FFT/window/hop, mel range, log offset, normalisation
* `mel_filterbank.json` — the exact mel matrix, so no filterbank has to be re-derived on device
* `labels.txt` — class order, one label per line
* `labels_phrases.json` — class index → phrase id → Arabic text

`README.md` has the matching Kotlin front-end.

In [ ]:
filterbank_path = extractor.save_filterbank(paths.exports_path / "mel_filterbank.json")
bundle.filterbank_path = filterbank_path

print("labels     :", bundle.labels_path)
print("metadata   :", bundle.metadata_path)
print("filterbank :", filterbank_path)
print()
for key, value in extractor.metadata().items():
    print("%-14s %s" % (key, value))


## 6 · What to ship

The recommendation is the smallest variant that still agrees with the Keras model. Override it if a
device measurement says otherwise.

In [ ]:
recommended = bundle.recommended()
if recommended is None:
    print("no variant passed verification — re-check the calibration clips and re-run")
else:
    print("recommended :", recommended.name)
    print("file        :", recommended.path)
    print("size        : %.2f MB" % (recommended.benchmark.size_kb / 1024.0))
    print("latency     : %.2f ms mean on this machine" % recommended.benchmark.mean_latency_ms)
    if recommended.verification:
        print("agreement   : %.2f%% with Keras" % (recommended.verification.agreement * 100))

print()
print("exports on Drive:")
for path in sorted(paths.exports_path.rglob("*")):
    if path.is_file():
        print("  %-34s %8.1f KB" % (path.name, path.stat().st_size / 1024.0))


## 7 · Android integration

Copy into `app/src/main/assets/`:

```
dhikr_int8.tflite      (or the recommended variant)
labels.txt
model_meta.json
mel_filterbank.json
```

On device, per inference:

1. record 16 kHz mono PCM16 into a ring buffer,
2. take the last `clip_seconds` of audio (`clip_samples` samples),
3. trim / normalise exactly as `model_meta.json.audio` describes,
4. compute the log mel spectrogram with the parameters in `model_meta.json.frontend`,
5. feed `(frames, n_mels, 1)` to the interpreter,
6. reject predictions below the threshold chosen in notebook 04, and debounce repeats so one spoken
   phrase counts once.

The full Kotlin front-end, the Gradle dependency and the threshold/debounce guidance are in
`README.md` under *Integrate into Android*.

In [ ]:
print("Export complete.\n")
print("copy to app/src/main/assets/:")
for name in ["labels.txt", "model_meta.json", "mel_filterbank.json"]:
    print("  ", paths.exports_path / name)
if recommended is not None:
    print("  ", recommended.path)
print()
print("Growing the dataset later: add recordings to dataset/<class>/ and re-run notebooks 01 → 05.")
print("Preprocessing skips clips it has already written, so re-runs only cost the new files.")
